# Optical Digits Classification

5,620 handwritten digits, each as an 8x8 image flattened into 64 pixel values (0-16). Predict the digit (0-9) from the pixels alone — the classic small-image benchmark that predates MNIST.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Optical Recognition of Handwritten Digits (id 80)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/80/data.csv')))
print(df.shape)
print('Samples per digit:')
print(df['class'].value_counts().sort_index())
df.head(3)


(5620, 65)
Samples per digit:
class
0    554
1    571
2    557
3    572
4    568
5    558
6    558
7    566
8    554
9    562
Name: count, dtype: int64


,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,...,Attribute56,Attribute57,Attribute58,Attribute59,Attribute60,Attribute61,Attribute62,Attribute63,Attribute64,class
0,0,1,6,15,12,1,0,0,0,7,...,0,0,0,6,14,7,1,0,0,0
1,0,0,10,16,6,0,0,0,0,7,...,0,0,0,10,16,15,3,0,0,0
2,0,0,8,15,16,13,0,0,0,1,...,0,0,0,9,14,0,0,0,0,7


In [2]:

feats = [c for c in df.columns if c != 'class']
X = df[feats]; y = df['class']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.9893
Logistic Regression accuracy: 0.9715
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       138
           1       0.99      1.00      1.00       143
           2       1.00      1.00      1.00       139
           3       0.99      0.99      0.99       143
           4       0.99      0.99      0.99       142
           5       1.00      0.98      0.99       139
           6       1.00      0.99      1.00       140
           7       0.98      0.99      0.99       142
           8       0.99      0.99      0.99       139
           9       0.96      0.96      0.96       140

    accuracy                           0.99      1405
   macro avg       0.99      0.99      0.99      1405
weighted avg       0.99      0.99      0.99      1405



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp.head(20))
imp.head(20).plot(kind='barh', title='Top 20 pixels for digit recognition')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


Attribute22    0.053039
Attribute43    0.045643
Attribute44    0.042757
Attribute37    0.039478
Attribute29    0.034839
Attribute27    0.034733
Attribute21    0.033381
Attribute31    0.032534
Attribute19    0.032393
Attribute20    0.029752
Attribute62    0.027597
Attribute30    0.026924
Attribute14    0.025695
Attribute35    0.025524
Attribute55    0.025319
Attribute61    0.025114
Attribute28    0.024378
Attribute11    0.024366
Attribute54    0.023103
Attribute39    0.021611
dtype: float64


Winner: Random Forest
